# 无约束排牙/工单排牙调用样例

In [17]:
# 导入必要的包以及定义函数
import os
import glob
import base64
import time
import requests
import json
import trimesh
import urllib
import numpy as np

def _create_colors():
    # 20 high contrast colors
    colors = [[230, 25, 75,255],[60, 180, 75,255],[255, 225, 25,255],\
            [0, 130, 200,255],[245, 130, 48,255],[145, 30, 180,255],[70, 240, 240,255],\
            [240, 50, 230,255],[210, 245, 60,255],[250, 190, 190,255],[0, 128, 128,255],\
            [230, 190, 255,255],[170, 110, 40,255],[255, 250, 200,255],[128, 0, 0,255],\
            [170, 255, 195,255],[128, 128, 0, 255]]
    #np.random.shuffle(colors)
    # gum color
    colors = [[255,255,255,255]] + colors
    return colors

def colored_mesh(mesh, label):
    COLORS = _create_colors()
    mcopy = mesh.copy()
    for i, l in enumerate(np.unique(label)):
        mcopy.visual.face_colors[np.where(label == l)[0]] = COLORS[i % 18]
    return mcopy

# 定义调用规则

请根据您从我方获取的信息修改以下代码块

In [18]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


In [19]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

## 无约束排牙

无约束排牙将牙齿自动排至符合美学的位置，主要用于营销展示与医患沟通。无约束指的是该算法不会考虑其他数据（如CBCT）提供的医学约束。https://www.chohotech.com/docs/cloud-zh/#/workflow/oral-arrangement-1

In [22]:
json_call = {
  "spec_group": "mesh-processing", # 调用的工作流组， 随API文档发送
  "spec_name": "oral-arrangement", # 调用的工作流名称， 随API文档发送, 
  "spec_version": "2.0-snapshot", # 调用的工作流版本，随API文档发送
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
        "upper_mesh": {"type": "drc", "data": upload_file('upper_jaw_scan.drc')},
        "lower_mesh": {"type": "ply", "data": upload_file('lower_jaw_scan.ply')},
        "ipr": {"U": False, "L": False},  # 可选
        "remove_teeth_set": [],  # 可选
        "gap": []  # 可选
      },
      "output_config": {
        "teeth_comp": {"type": "ply"},
        "align_matrix": {},  # 如果需要
        "transformation_dict": {}  # 如果需要
      }
    }
result = run_job_and_get_results(json_call, 300)

workflow id is wf_1766568393-a6fad113-8b9c-4378-b2f8-1193ba9d962b
API finished in 63.30111789703369s


In [23]:
print(f"输出包含的键: {list(result.keys())}")

输出包含的键: ['align_matrix', 'teeth_comp', 'transformation_dict', 'u_align_matrix']


In [ ]:
#将输出的teeth_comp中每颗牙齿按transformation_dict变换可以得到排牙后的结果，下面是teeth_comp的结果 ：
sum([retrieve_mesh(result['teeth_comp'][k]) for k in result['teeth_comp'].keys()], None).show()

## 工单排牙

工单排牙将牙齿自动排至符合医生工单要求的位置。https://www.chohotech.com/docs/cloud-zh/#/workflow/oral-arrangement-with-form-1

您可以上传json格式的工单也可以手动填写，如下：

In [ ]:
def example_doctor_form():  """工单排牙示例"""
    # 工单配置
doctor_form = {
        "type": "U+L",
        "locked_teeth_set": [],
        "middle_line_position": {
            "U": ["U", "keep", 0],
            "L": ["L", "keep", 0]
        },
        "front_y_axis_position": None,
        "back_y_axis_position": None,
        "z_axis_position": {
            "front_teeth": ["keep", 0.0],
            "back_teeth": ["keep", 0.0]
        },
        "x_axis_position": None,
        "collision_removal": {
            "U": {"1": [], "0": []},
            "L": {"1": [], "0": []}
        },
        "init_gap": {},
        "gap": {},
        "ipr": {},
        "remove_teeth_set": [],
        "y_axis_relative_position": {
            "left": {"canine": "I", "molar": "I"},
            "right": {"canine": "I", "molar": "I"}
        },
        "front_y_axis_relative_position": ["standard", {"U": "any", "L": "any"}],
        "z_axis_relative_position": ["standard", {"front": "any", "back": "any"}],
        "back_teeth_x_axis_relative_position": False,
        "middle_line_opt": {"U": True, "L": True}
    }

In [ ]:
json_call = {
  "spec_group": "mesh-processing", # 调用的工作流组， 随API文档发送
  "spec_name": "oral-arrangement-with-form", # 调用的工作流名称， 随API文档发送, 
  "spec_version": "1.0-snapshot", # 调用的工作流版本，随API文档发送
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
        "upper_mesh": {"type": "drc", "data": upload_file('upper_jaw_scan.drc')},
        "lower_mesh": {"type": "ply", "data": upload_file('lower_jaw_scan.ply')},
        "form": json.dumps(doctor_form)  # 工单必须是JSON字符串
      },
  "output_config": {
        "teeth_comp": {"type": "ply"},        # 牙齿补全后
        "arranged_comp": {"type": "ply"},     # 排牙后牙齿
        "upper_mesh": {"type": "ply"},        # 预处理后的上颌
        "lower_mesh": {"type": "ply"},        # 预处理后的下颌
        "u_align_matrix": {},                 # 上颌对齐矩阵
        "l_align_matrix": {},                 # 下颌对齐矩阵
        "transformation_dict": {},            # 单个牙齿变换矩阵
        "form": {}                            # 返回的工单配置
  }
}
result = run_job_and_get_results(json_call, 300)

In [ ]:
print(f"输出包含的键: {list(result.keys())}")

In [ ]:
# 显示排牙结果
sum([retrieve_mesh(result['arranged_comp'][k]) for k in result['arranged_comp'].keys()], None).show()